In [3]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Cargar variables de entorno
load_dotenv()

# Configuración del modelo de Embeddings (el estándar de la industria)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Directorio donde se guardará la base de datos local
CHROMA_PATH = "chroma_db"

In [5]:
def create_vector_store(domain_name):
    print(f"--- Procesando dominio: {domain_name} ---")
    
    # 1. Cargar documentos
    loader = DirectoryLoader(
        f"./data/{domain_name}_docs", 
        glob="**/*.md", 
        loader_cls=UnstructuredMarkdownLoader
    )
    docs = loader.load()
    print(f"Documentos cargados: {len(docs)}")

    # 2. Dividir en fragmentos (Chunks)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        add_start_index=True
    )
    chunks = text_splitter.split_documents(docs)
    print(f"Fragmentos creados: {len(chunks)}")

    # 3. Guardar en ChromaDB (una colección por dominio)
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=CHROMA_PATH,
        collection_name=f"{domain_name}_collection"
    )
    
    print(f"✅ Dominio {domain_name} indexado correctamente.\n")
    return vector_store

# Ejecutamos la creación para nuestros 3 dominios
booking_db = create_vector_store("booking")
logistics_db = create_vector_store("logistics")
marketing_db = create_vector_store("marketing")

Error loading file data\booking_docs\doc_0.md


--- Procesando dominio: booking ---


ModuleNotFoundError: No module named 'markdown'

In [ ]:
# Prueba de búsqueda semántica en el dominio de Booking
query = "Busco un bar en Córdoba para tocar rock con mi banda, somos 4 personas"

# Buscamos los 3 fragmentos más parecidos
results = booking_db.similarity_search(query, k=3)

print(f"Pregunta: {query}\n")
print("--- Resultados encontrados ---")
for i, res in enumerate(results):
    print(f"\nResultado {i+1}:")
    print(res.page_content[:300] + "...") # Mostramos solo el inicio del fragmento